# 4.2.3 Learning Rate Strategies

Constant, step decay, cosine, warmup+cosine — visualise LR curves and training effect.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def step(e,base=.1,drop=.5,every=50): return base*(drop**(e//every))
def cosine(e,base=.1,T=200): return base*.5*(1+np.cos(np.pi*e/T))
def warmup_cos(e,base=.1,w=20,T=200):
    return base*e/w if e<w else base*.5*(1+np.cos(np.pi*(e-w)/(T-w)))

eps=np.arange(200)
for name,fn in [('Constant',lambda e:.1),('Step',step),('Cosine',cosine),('Warmup+Cos',warmup_cos)]:
    plt.plot(eps,[fn(e) for e in eps],label=name)
plt.xlabel('Epoch'); plt.title('LR Schedules'); plt.legend(); plt.show()

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
X,y=make_moons(500,noise=.2,random_state=42); Xs=StandardScaler().fit_transform(X)
sigmoid=lambda x:1/(1+np.exp(-np.clip(x,-500,500))); relu=lambda x:np.maximum(0,x); relu_d=lambda x:(x>0).astype(float)

for sname,lr_fn in [('Constant',lambda e:.05),('Warmup+Cos',warmup_cos)]:
    rng=np.random.default_rng(42)
    W1=rng.standard_normal((2,32))*np.sqrt(2/2); b1=np.zeros(32)
    W2=rng.standard_normal((32,1))*np.sqrt(2/32); b2=np.zeros(1)
    ls=[]
    for e in range(200):
        lr=lr_fn(e); z1=Xs@W1+b1; a1=relu(z1); z2=a1@W2+b2; a2=sigmoid(z2)
        y2=y.reshape(-1,1); p=np.clip(a2,1e-7,1-1e-7); n=len(y)
        ls.append(-np.mean(y2*np.log(p)+(1-y2)*np.log(1-p)))
        dz2=(a2-y2)/n; dz1=(dz2@W2.T)*relu_d(z1)
        W2-=lr*(a1.T@dz2); b2-=lr*dz2.sum(0); W1-=lr*(Xs.T@dz1); b1-=lr*dz1.sum(0)
    plt.plot(ls,label=sname); print(f'{sname}: {ls[-1]:.4f}')
plt.legend(); plt.title('Schedule Effect'); plt.show()